# FAISS Index — Build & Test Semantic Search

**What this does:** Build a FAISS index from question embeddings and test the full recommendation pipeline

**Input:**
- `question_embeddings.npy` — (498644, 384) vectors
- `question_ids.npy` — ID mapping
- `questions_cleaned.parquet` — question details
- `answers_cleaned.parquet` — answer details

**Output:**
- `faiss_index.bin` — FAISS index file (reusable, no need to rebuild)

**Runtime:** Index builds in ~5 seconds. Search takes ~3ms per query.

---
## 1. Setup

In [1]:
!pip install faiss-cpu sentence-transformers pyarrow -q

In [2]:
import numpy as np
import pandas as pd
import faiss
import time
import os
from sentence_transformers import SentenceTransformer

print(f'FAISS version: {faiss.__version__}')
print('Libraries imported!')

FAISS version: 1.13.2
Libraries imported!


---
## 2. Load Everything

In [3]:

DATA_DIR = '../Dataset_Cleaned'

# Load embeddings
print('Loading embeddings...')
embeddings = np.load(os.path.join(DATA_DIR, 'question_embeddings.npy'))
print(f'  Embeddings: {embeddings.shape}')

# Load ID mapping
question_ids = np.load(os.path.join(DATA_DIR, 'question_ids.npy'))
print(f'  Question IDs: {len(question_ids):,}')

# Load question details
print('Loading questions...')
questions_df = pd.read_parquet(os.path.join(DATA_DIR, 'questions_cleaned.parquet'))
print(f'  Questions: {len(questions_df):,} rows')

# Load answer details
print('Loading answers...')
answers_df = pd.read_parquet(os.path.join(DATA_DIR, 'answers_cleaned.parquet'))
print(f'  Answers: {len(answers_df):,} rows')

# Load the embedding model (needed to encode user queries)
print('Loading model...')
model = SentenceTransformer('all-MiniLM-L6-v2')
print(f'  Model: all-MiniLM-L6-v2')

print(f'\nAll loaded!')

Loading embeddings...
  Embeddings: (498644, 384)
  Question IDs: 498,644
Loading questions...
  Questions: 498,644 rows
Loading answers...
  Answers: 1,199,383 rows
Loading model...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


  Model: all-MiniLM-L6-v2

All loaded!


---
## 3. Build FAISS Index

We use `IndexFlatIP` (Inner Product) which computes cosine similarity when vectors are normalized. This is a brute-force index — it compares the query with all 498K vectors. At our scale this takes ~3ms per query, which is fast enough.

In [4]:
# Step 1: Normalize embeddings (required for cosine similarity via Inner Product)
# After normalization, Inner Product = Cosine Similarity
print('Normalizing embeddings...')
faiss.normalize_L2(embeddings)

# Step 2: Build the index
dimension = embeddings.shape[1]  # 384
print(f'Building FAISS index (dimension={dimension})...')

start = time.time()
index = faiss.IndexFlatIP(dimension)  # IP = Inner Product (cosine after normalization)
index.add(embeddings)                 # Add all 498K vectors
elapsed = time.time() - start

print(f'\nIndex built in {elapsed:.2f}s')
print(f'  Total vectors: {index.ntotal:,}')
print(f'  Dimension: {dimension}')
print(f'  Index type: Flat (brute-force, exact results)')

Normalizing embeddings...
Building FAISS index (dimension=384)...

Index built in 0.32s
  Total vectors: 498,644
  Dimension: 384
  Index type: Flat (brute-force, exact results)


In [5]:
# Save the index
index_path = os.path.join(DATA_DIR, 'faiss_index.bin')
faiss.write_index(index, index_path)

idx_size = os.path.getsize(index_path) / (1024**2)
print(f'Saved: faiss_index.bin ({idx_size:.1f} MB)')

Saved: faiss_index.bin (730.4 MB)


---
## 4. Build the Search Function

This is the core of our recommendation system. Given a user query, it:
1. Converts query to embedding
2. Finds similar questions via FAISS
3. Looks up question details + best answer
4. Returns ranked results

In [6]:
def search(query, top_k=5):
    """
    Search for similar Stack Overflow questions.
    
    Args:
        query: User's search query (plain text)
        top_k: Number of results to return
    
    Returns:
        List of dicts with question details + top answer
    """
    # Step 1: Convert query to embedding
    query_vector = model.encode([query], convert_to_numpy=True)
    faiss.normalize_L2(query_vector)  # Must normalize (same as index vectors)
    
    # Step 2: Search FAISS
    scores, indices = index.search(query_vector, top_k)
    
    # Step 3: Build results
    results = []
    for rank, (score, idx) in enumerate(zip(scores[0], indices[0]), 1):
        
        # Get question ID from our mapping (safe lookup)
        q_id = int(question_ids[idx])
        
        # Get question details
        question = questions_df[questions_df['id'] == q_id].iloc[0]
        
        # Get the best answer for this question
        q_answers = answers_df[answers_df['question_id'] == q_id].sort_values('answer_rank')
        top_answer = q_answers.iloc[0] if len(q_answers) > 0 else None
        
        results.append({
            'rank': rank,
            'similarity': round(float(score), 3),
            'question_id': q_id,
            'title': question['title'],
            'tags': question['tags'],
            'score': int(question['score']),
            'views': int(question['view_count']),
            'has_accepted': bool(question['has_accepted_answer']),
            'answer_body': top_answer['body'][:300] + '...' if top_answer is not None else 'No answer',
            'answer_score': int(top_answer['score']) if top_answer is not None else 0,
            'answer_accepted': bool(top_answer['is_accepted']) if top_answer is not None else False
        })
    
    return results


def display_results(query, results):
    """Pretty-print search results."""
    print(f'\nQuery: "{query}"')
    print('=' * 80)
    
    for r in results:
        accepted = ' ACCEPTED' if r['answer_accepted'] else ''
        print(f'\n  #{r["rank"]} (similarity: {r["similarity"]:.3f})')
        print(f'  {r["title"]}')
        print(f'     Tags: {r["tags"]}  |  Score: {r["score"]:,}  |  Views: {r["views"]:,}')
        print(f'     Top Answer (score: {r["answer_score"]:,}){accepted}:')
        print(f'        {r["answer_body"][:200]}...')
    
    print('\n' + '=' * 80)


print('Search function ready!')

Search function ready!


---
## 5. Test the Search

In [7]:
# Test 1: Python sorting
results = search('how to sort a list in python', top_k=5)
display_results('how to sort a list in python', results)


Query: "how to sort a list in python"

  #1 (similarity: 0.855)
  How to sort python list of strings of numbers
     Tags: python  |  Score: 35  |  Views: 86,012
     Top Answer (score: 88) ACCEPTED:
        You want to sort based on the  float  values (not string values), so try:......

  #2 (similarity: 0.854)
  Sorting list of lists by the first element of each sub-list
     Tags: python  |  Score: 52  |  Views: 65,540
     Top Answer (score: 66):
        Use sorted function along with passing anonymous function as value to the key argument.  key=lambda x: x[0]  will do sorting according to the first element in each sublist.......

  #3 (similarity: 0.850)
  Python list sort in descending order
     Tags: python,sorting,reverse  |  Score: 419  |  Views: 771,010
     Top Answer (score: 503):
        This will give you a sorted version of the array. 
 
 If you want to sort in-place: 
 
 Check the docs at  Sorting HOW TO......

  #4 (similarity: 0.846)
  Sort python list by function
 

In [8]:
# Test 2: JavaScript async/await
results = search('handle async await errors in javascript', top_k=5)
display_results('handle async await errors in javascript', results)


Query: "handle async await errors in javascript"

  #1 (similarity: 0.787)
  async/await catch rejected Promises
     Tags: javascript,node.js,babeljs  |  Score: 18  |  Views: 14,643
     Top Answer (score: 21) ACCEPTED:
        What  bad;  alone is supposed to do? The error is caught as expected, you just don't do anything with it: 
 
 This outputs  bad  as expected.  Code here .......

  #2 (similarity: 0.771)
  Can I throw error in an async function?
     Tags: javascript,promise,async-await,es6-promise  |  Score: 31  |  Views: 39,784
     Top Answer (score: 31) ACCEPTED:
        I would do: 
 
 But I think it comes to personal preference I guess? You could always return  Promise.reject(new Error(error)); .......

  #3 (similarity: 0.768)
  Catching errors from nested async/await functions
     Tags: javascript,node.js,async-await,ecmascript-2017  |  Score: 14  |  Views: 7,655
     Top Answer (score: 7) ACCEPTED:
        If  etcFunction()  throws, does the error bubble up all the w

In [9]:
# Test 3: SQL joins
results = search('how to join two tables in SQL', top_k=5)
display_results('how to join two tables in SQL', results)


Query: "how to join two tables in SQL"

  #1 (similarity: 0.808)
  how to join more than 2 table using sql Query?
     Tags: sql  |  Score: 7  |  Views: 22,699
     Top Answer (score: 11):
        example: 
 
 If you wanted all records from table1 even without a matching record in another table, then the join should be "left join" instead.......

  #2 (similarity: 0.769)
  Join two tables with multiple foreign keys
     Tags: sql  |  Score: 7  |  Views: 13,291
     Top Answer (score: 3):
        You can try this......

  #3 (similarity: 0.754)
  Multiple column left join from table1, table2
     Tags: mysql  |  Score: 8  |  Views: 696
     Top Answer (score: 3):
        Click the link below for a running demo. 
 SQLFiddle......

  #4 (similarity: 0.749)
  sql join two table
     Tags: mysql,sql,tsql,left-join  |  Score: 11  |  Views: 20,474
     Top Answer (score: 20) ACCEPTED:
        You can write left outer join between this two tables Best way to understand is check the below imag

In [10]:
# Test 4: Semantic understanding test
# This query uses DIFFERENT WORDS than any Stack Overflow title
# But the model should understand the MEANING and find relevant results
results = search('why is my program running slowly with large data', top_k=5)
display_results('why is my program running slowly with large data', results)


Query: "why is my program running slowly with large data"

  #1 (similarity: 0.607)
  Reading a huge .csv file
     Tags: python,python-2.7,file,csv  |  Score: 141  |  Views: 239,823
     Top Answer (score: 180) ACCEPTED:
        You are reading all rows into a list, then processing that list.  Don't do that . 
 Process your rows as you produce them. If you need to filter the data first, use a generator function: 
 
 I also si...

  #2 (similarity: 0.601)
  Java program is getting slower after running for a while
     Tags: java,linux,performance,memory,garbage-collection  |  Score: 19  |  Views: 5,079
     Top Answer (score: 1):
        I'm sorry to post this as an answer but I don't have enough score to comment. 
 If you think it's a GC related issue I'd change it for the Garbage 1 Collector –XX:+UseG1GC 
 I found this brief explana...

  #3 (similarity: 0.593)
  Files loading slower on second run of application, with repro code
     Tags: c++,performance,visual-studio-2013,windows-

In [11]:
# Test 5: CSS layout question
results = search('center a div vertically and horizontally', top_k=5)
display_results('center a div vertically and horizontally', results)


Query: "center a div vertically and horizontally"

  #1 (similarity: 0.891)
  How to center a div vertically?
     Tags: html,css  |  Score: 18  |  Views: 34,366
     Top Answer (score: 37) ACCEPTED:
        If you only have to support browsers that support  transform  (or its vendor prefixed versions), use this one weird old trick to vertically align elements. 
 
 If you have to support older browsers, y...

  #2 (similarity: 0.883)
  How do I center content in a div using CSS?
     Tags: css  |  Score: 40  |  Views: 151,758
     Top Answer (score: 21):
        To align horizontally it's pretty straight forward: 
 
 and for vertical align, it's a bit tricky: 
here's the  source......

  #3 (similarity: 0.839)
  How to center horizontally div inside parent div
     Tags: html,css  |  Score: 131  |  Views: 209,704
     Top Answer (score: 188) ACCEPTED:
        I am assuming the parent div has no width or a wide width, and the child div has a smaller width. The following will set the ma

---
## 6. Speed Benchmark

In [12]:
# Measure search speed
test_queries = [
    'how to read a file in python',
    'what is a closure in javascript',
    'difference between inner join and left join',
    'how to center a div in css',
    'what is dependency injection in java'
]

print('Speed Benchmark (5 queries):')
print('-' * 50)

times = []
for q in test_queries:
    start = time.time()
    query_vector = model.encode([q], convert_to_numpy=True)
    faiss.normalize_L2(query_vector)
    scores, indices = index.search(query_vector, 10)
    elapsed = (time.time() - start) * 1000  # ms
    times.append(elapsed)
    print(f'  {elapsed:>6.1f} ms  |  "{q[:50]}"')

print('-' * 50)
print(f'  Average: {np.mean(times):.1f} ms per query')
print(f'  (includes model encoding + FAISS search)')

Speed Benchmark (5 queries):
--------------------------------------------------
    59.5 ms  |  "how to read a file in python"
    66.4 ms  |  "what is a closure in javascript"
    56.2 ms  |  "difference between inner join and left join"
    53.1 ms  |  "how to center a div in css"
    57.0 ms  |  "what is dependency injection in java"
--------------------------------------------------
  Average: 58.4 ms per query
  (includes model encoding + FAISS search)


---
## 7. How to Load Everything Later (for Backend)

When we build the FastAPI backend, this is all we need to load:

In [13]:
# This cell shows what the backend will do at startup.
# You don't need to run this now — it's here for reference.

print('Backend startup code (for reference):')
print('=' * 50)
print('''
import faiss
import numpy as np
import pandas as pd
from sentence_transformers import SentenceTransformer

# Load once at startup (~10 seconds)
model = SentenceTransformer('all-MiniLM-L6-v2')
index = faiss.read_index('faiss_index.bin')
question_ids = np.load('question_ids.npy')
questions_df = pd.read_parquet('questions_cleaned.parquet')
answers_df = pd.read_parquet('answers_cleaned.parquet')

# Then for each user query (~20ms):
query_vec = model.encode([user_query])
faiss.normalize_L2(query_vec)
scores, indices = index.search(query_vec, 10)
# ... lookup questions and answers by ID ...
''')

Backend startup code (for reference):

import faiss
import numpy as np
import pandas as pd
from sentence_transformers import SentenceTransformer

# Load once at startup (~10 seconds)
model = SentenceTransformer('all-MiniLM-L6-v2')
index = faiss.read_index('faiss_index.bin')
question_ids = np.load('question_ids.npy')
questions_df = pd.read_parquet('questions_cleaned.parquet')
answers_df = pd.read_parquet('answers_cleaned.parquet')

# Then for each user query (~20ms):
query_vec = model.encode([user_query])
faiss.normalize_L2(query_vec)
scores, indices = index.search(query_vec, 10)
# ... lookup questions and answers by ID ...



In [14]:
print('=' * 60)
print('FAISS INDEX COMPLETE')
print('=' * 60)
print(f'\n  Index vectors:    {index.ntotal:,}')
print(f'  Dimension:        {dimension}')
print(f'  Search speed:     ~{np.mean(times):.0f}ms per query')
print(f'  Index type:       Flat (exact cosine similarity)')
print(f'\n  Files in {DATA_DIR}/:')
print(f'    faiss_index.bin        ({idx_size:.1f} MB)')
print(f'    question_embeddings.npy (from previous notebook)')
print(f'    question_ids.npy       (from previous notebook)')
print(f'\n  Pipeline so far:')
print(f'    Done: Data Extraction (BigQuery)')
print(f'    Done: Data Cleaning (HTML to clean text)')
print(f'    Done: Embedding Generation (MiniLM)')
print(f'    Done: FAISS Index (this notebook)')
print(f'    Next: FastAPI backend')
print('=' * 60)

FAISS INDEX COMPLETE

  Index vectors:    498,644
  Dimension:        384
  Search speed:     ~58ms per query
  Index type:       Flat (exact cosine similarity)

  Files in ../Dataset_Cleaned/:
    faiss_index.bin        (730.4 MB)
    question_embeddings.npy (from previous notebook)
    question_ids.npy       (from previous notebook)

  Pipeline so far:
    Done: Data Extraction (BigQuery)
    Done: Data Cleaning (HTML to clean text)
    Done: Embedding Generation (MiniLM)
    Done: FAISS Index (this notebook)
    Next: FastAPI backend
